# Fraud Detection — Layer 1
## Notebook 2: Preprocessing
**Author:** Frederick Amartey-Fio  
**Institution:** JUNIA ISEN — MSc Big Data  
**Date:** May 2026

---

### Objective
Prepare the dataset for machine learning.  
We will:
- Scale the Amount and Time features
- Split the data into training and test sets
- Handle the severe class imbalance using SMOTE

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — IMPORTS
# Load all libraries needed for preprocessing.
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing tools
from sklearn.preprocessing import StandardScaler      # scales features to same range
from sklearn.model_selection import train_test_split  # splits data into train and test

# Handling class imbalance
from imblearn.over_sampling import SMOTE  # creates synthetic fraud samples

# Display plots inline
%matplotlib inline
sns.set_style("whitegrid")

print("Libraries loaded.")

Libraries loaded.


In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — LOAD DATASET
# Load the same dataset we explored in Notebook 1.
# ─────────────────────────────────────────────────────────────

df = pd.read_csv("../data/creditcard.csv")

# Quick confirmation
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Fraud cases   : {df['Class'].sum():,}")
print(f"Legit cases   : {(df['Class'] == 0).sum():,}")

Dataset loaded: 284,807 rows, 31 columns
Fraud cases   : 492
Legit cases   : 284,315


In [3]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — SCALING AMOUNT AND TIME
# The V1-V28 features are already scaled (they came from PCA).
# But Amount ranges from $0 to $25,691 and Time is in seconds.
# Large unscaled values confuse machine learning models.
# We use StandardScaler to bring them to the same range as V features.
# ─────────────────────────────────────────────────────────────

scaler = StandardScaler()

# Scale Amount and Time into new columns
df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
df["Time_scaled"] = scaler.fit_transform(df[["Time"]])

# Drop the original unscaled columns — we no longer need them
df = df.drop(columns=["Amount", "Time"])

# Confirm
print("Scaling complete.")
print(f"Amount_scaled — mean: {df['Amount_scaled'].mean():.4f}, std: {df['Amount_scaled'].std():.4f}")
print(f"Time_scaled   — mean: {df['Time_scaled'].mean():.4f}, std: {df['Time_scaled'].std():.4f}")
print()
print(f"Dataset shape after scaling: {df.shape}")

Scaling complete.
Amount_scaled — mean: -0.0000, std: 1.0000
Time_scaled   — mean: -0.0000, std: 1.0000

Dataset shape after scaling: (284807, 31)


In [4]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — DEFINE FEATURES AND TARGET
# X = everything the model learns from (the input)
# y = what the model is trying to predict (fraud or not)
# ─────────────────────────────────────────────────────────────

# X is every column except Class
X = df.drop(columns=["Class"])

# y is the Class column — 0 for legit, 1 for fraud
y = df["Class"]

print(f"Features (X) shape : {X.shape}")
print(f"Target (y) shape   : {y.shape}")
print()
print(f"Feature columns ({len(X.columns)}):")
print(list(X.columns))

Features (X) shape : (284807, 30)
Target (y) shape   : (284807,)

Feature columns (30):
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_scaled', 'Time_scaled']


In [5]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — TRAIN / TEST SPLIT
# We split the data into two sets:
# - Training set (80%) → the model learns from this
# - Test set (20%)     → we evaluate the model on this
#
# stratify=y is critical here — it ensures both sets contain
# the same fraud ratio (0.17%). Without it, the test set could
# accidentally end up with very few or no fraud cases.
# ─────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% goes to test set
    random_state=42,     # ensures same split every time you run
    stratify=y           # preserves fraud ratio in both sets
)

print(f"Training set   : {X_train.shape[0]:,} rows")
print(f"Test set       : {X_test.shape[0]:,} rows")
print()
print(f"Fraud in training set : {y_train.sum():,} ({y_train.mean()*100:.3f}%)")
print(f"Fraud in test set     : {y_test.sum():,} ({y_test.mean()*100:.3f}%)")

Training set   : 227,845 rows
Test set       : 56,962 rows

Fraud in training set : 394 (0.173%)
Fraud in test set     : 98 (0.172%)


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — SMOTE (Synthetic Minority Oversampling Technique)
# We have 394 fraud cases vs 227,451 legit in the training set.
# SMOTE creates synthetic fraud samples by interpolating between
# existing fraud cases — giving the model enough fraud examples
# to learn from.
#
# CRITICAL RULE: We apply SMOTE only on the TRAINING set.
# Never on the test set — the test set must reflect reality.
# ─────────────────────────────────────────────────────────────

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(f"  Legit cases : {(y_train == 0).sum():,}")
print(f"  Fraud cases : {(y_train == 1).sum():,}")
print()
print("After SMOTE:")
print(f"  Legit cases : {(y_train_smote == 0).sum():,}")
print(f"  Fraud cases : {(y_train_smote == 1).sum():,}")
print()
print(f"Training set size went from {len(y_train):,} to {len(y_train_smote):,} rows")

Before SMOTE:
  Legit cases : 227,451
  Fraud cases : 394

After SMOTE:
  Legit cases : 227,451
  Fraud cases : 227,451

Training set size went from 227,845 to 454,902 rows


In [7]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — SAVE PROCESSED DATA
# Save all arrays to disk so Notebook 3 (modelling) can load
# them directly without rerunning preprocessing every time.
# ─────────────────────────────────────────────────────────────

import joblib

# Save the SMOTE-balanced training data
joblib.dump(X_train_smote, "../models/X_train_smote.pkl")
joblib.dump(y_train_smote, "../models/y_train_smote.pkl")

# Save the original test data (no SMOTE — reflects real world)
joblib.dump(X_test, "../models/X_test.pkl")
joblib.dump(y_test, "../models/y_test.pkl")

# Save feature column names — critical for the Streamlit app later
joblib.dump(list(X.columns), "../models/feature_columns.pkl")

print("Saved successfully:")
print("  models/X_train_smote.pkl")
print("  models/y_train_smote.pkl")
print("  models/X_test.pkl")
print("  models/y_test.pkl")
print("  models/feature_columns.pkl")

Saved successfully:
  models/X_train_smote.pkl
  models/y_train_smote.pkl
  models/X_test.pkl
  models/y_test.pkl
  models/feature_columns.pkl


## Preprocessing Conclusions

| Step | Decision | Reason |
|---|---|---|
| Scale Amount and Time | StandardScaler | V features already scaled — must match their range |
| Train/test split | 80/20 with stratify=y | Preserves 0.17% fraud ratio in both sets |
| SMOTE | Applied to training set only | Balances fraud/legit from 394 vs 227,451 to 227,451 vs 227,451 |
| Save to disk | joblib .pkl files | Notebook 3 loads directly — no reprocessing needed |

**Next step:** Notebook 3 — train and compare three models: Logistic Regression, XGBoost, and Isolation Forest.